In [1]:
from pathlib import Path
import random
import shutil

SOURCE_DIR = Path("./animals")
OUTPUT_DIR = Path("./animals_dataset")

# Dataset percentages
TRAIN_RATIO = 0.70
DEV_RATIO = 0.15
TEST_RATIO = 0.15

# Makes the split repeatable
RANDOM_SEED = 42

# Supported image types
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

In [2]:
if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Source dataset was not found: {SOURCE_DIR.resolve()}"
    )

if SOURCE_DIR.resolve() == OUTPUT_DIR.resolve():
    raise ValueError(
        "SOURCE_DIR and OUTPUT_DIR must be different folders."
    )

total_ratio = TRAIN_RATIO + DEV_RATIO + TEST_RATIO

if abs(total_ratio - 1.0) > 0.0001:
    raise ValueError(
        "TRAIN_RATIO, DEV_RATIO, and TEST_RATIO must add up to 1."
    )


if OUTPUT_DIR.exists():
    print(f"Deleting existing folder: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class_folders = [
    folder
    for folder in SOURCE_DIR.iterdir()
    if folder.is_dir()
]

if not class_folders:
    raise ValueError(
        f"No class folders were found inside {SOURCE_DIR.resolve()}"
    )

random_generator = random.Random(RANDOM_SEED)

for class_folder in sorted(class_folders):

    class_name = class_folder.name

    images = [
        image
        for image in class_folder.iterdir()
        if image.is_file()
        and image.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if not images:
        print(f"No images found for class: {class_name}")
        continue

    # Shuffle the images
    random_generator.shuffle(images)

    number_of_images = len(images)

    train_count = int(number_of_images * TRAIN_RATIO)
    dev_count = int(number_of_images * DEV_RATIO)

    train_images = images[:train_count]

    dev_images = images[
        train_count:
        train_count + dev_count
    ]

    test_images = images[
        train_count + dev_count:
    ]

    split_groups = {
        "train": train_images,
        "dev": dev_images,
        "test": test_images,
    }

    # Copy images into their new folders
    for split_name, split_images in split_groups.items():

        destination_folder = (
            OUTPUT_DIR
            / split_name
            / class_name
        )

        destination_folder.mkdir(
            parents=True,
            exist_ok=True,
        )

        for image_path in split_images:
            shutil.copy2(
                image_path,
                destination_folder / image_path.name,
            )

    print(f"\nClass: {class_name}")
    print(f"Total: {number_of_images}")
    print(f"Training: {len(train_images)}")
    print(f"Development: {len(dev_images)}")
    print(f"Testing: {len(test_images)}")


print("\nDataset splitting completed.")
print(f"New dataset folder: {OUTPUT_DIR.resolve()}")


Class: cheetah
Total: 320
Training: 224
Development: 48
Testing: 48

Class: fox
Total: 237
Training: 165
Development: 35
Testing: 37

Class: hyena
Total: 292
Training: 204
Development: 43
Testing: 45

Class: lion
Total: 287
Training: 200
Development: 43
Testing: 44

Class: tiger
Total: 268
Training: 187
Development: 40
Testing: 41

Class: wolf
Total: 258
Training: 180
Development: 38
Testing: 40

Dataset splitting completed.
New dataset folder: \\mac\Home\Downloads\Week3_Deep Learning\animals_dataset
